In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# ▶ Install prerequisites (if not already)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet networkx numpy matplotlib

# ─────────────────────────────────────────────────────────────────────────────
# ▶ Generate a dataset of richly‐attributed hospital‐wing graphs
# ─────────────────────────────────────────────────────────────────────────────
import os, random
import networkx as nx
import numpy as np

def build_hospital_scenario(
    floors=3,
    rooms_per_floor=30,
    corridor_nodes=50,
    stair_nodes=20,
    corridor_edges=200,
    stair_edges=60,
    utility_nodes=15,
    hazard_nodes=30,
    agents=8,
    seed=None
):
    """Returns one directed NetworkX graph representing a quake‐damaged hospital."""
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    G = nx.DiGraph()
    # 1) Create room nodes
    for f in range(1, floors+1):
        for r in range(1, rooms_per_floor+1):
            nid = f"F{f}_R{r}"
            G.add_node(nid, **{
                "type":"room",
                "floor":f,
                "roomType": random.choice(["ward","ICU","OR","storage","lobby"]),
                "lighting": round(random.uniform(0,1),3),
                "temperature": round(random.uniform(15,35),1),
                "structuralIntegrity": round(random.uniform(0,1),3),
                "urgency": round(random.uniform(0,1),3),
                "gasLeak": random.choice([True,False]),
                "occupants": random.randint(0,4)
            })
    # 2) Corridor & Stair nodes
    for i in range(corridor_nodes):
        G.add_node(f"Corridor_{i}", type="corridor",
                   lighting=round(random.uniform(0,1),3))
    for i in range(stair_nodes):
        G.add_node(f"Stair_{i}", type="stair",
                   structuralIntegrity=round(random.uniform(0,1),3))
    # 3) Utility & Hazard nodes
    for i in range(utility_nodes):
        G.add_node(f"Utility_{i}", type="utility",
                   subsystem=random.choice(["power","comm","water","vent"]))
    for i in range(hazard_nodes):
        G.add_node(f"Hazard_{i}", type="hazard",
                   hazardType=random.choice(["gas","fire","collapse","radiation"]))
    all_rooms = [n for n,d in G.nodes(data=True) if d["type"]=="room"]
    # 4) Corridor edges between rooms
    for _ in range(corridor_edges):
        u,v = random.sample(all_rooms,2)
        G.add_edge(u, v, edgeType="corridor", **{
            "debris": round(random.uniform(0,5),2),
            "commReliability": round(random.uniform(0,1),3),
            "traversalTime": round(random.uniform(1,10),2),
            "structuralRisk": round(random.uniform(0,1),3),
            "trustScore": round(random.uniform(0,1),3),
            "radiation": round(random.uniform(0,0.2),3),
            "tempGradient": round(random.uniform(-5,5),2),
            "visibility": round(random.uniform(0,1),3)
        })
    # 5) Stair edges across floors
    for _ in range(stair_edges):
        f1,f2 = random.sample(range(1,floors+1),2)
        r1 = f"F{f1}_R{random.randint(1,rooms_per_floor)}"
        r2 = f"F{f2}_R{random.randint(1,rooms_per_floor)}"
        G.add_edge(r1, r2, edgeType="stair", **{
            "debris": round(random.uniform(0,2),2),
            "commReliability": round(random.uniform(0,1),3),
            "traversalTime": round(random.uniform(2,15),2),
            "structuralRisk": round(random.uniform(0,1),3),
            "trustScore": round(random.uniform(0,1),3),
            "radiation": round(random.uniform(0,0.1),3),
            "tempGradient": round(random.uniform(-3,3),2),
            "visibility": round(random.uniform(0,1),3)
        })
    # 6) Utility/hazard → room links
    special = [n for n in G.nodes if G.nodes[n]["type"] in ("utility","hazard")]
    for u in special:
        for _ in range(random.randint(1,3)):
            v = random.choice(all_rooms)
            G.add_edge(u, v, edgeType="service", weight=1.0)
    # 7) Robots
    for a in range(agents):
        nid = f"Robot_{a}"
        G.add_node(nid, type="agent",
                   battery=round(random.uniform(0,1),3),
                   sensorReliability=round(random.uniform(0.5,1),3),
                   role=random.choice(["rescue","medical","scout"]))
        start = random.choice(all_rooms)
        G.add_edge(nid, start, edgeType="start", traversalTime=0, trustScore=1.0)
    return G

# ─────────────────────────────────────────────────────────────────────────────
# ▶ Build & save a small “dataset” of 5 different scenarios
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs("hospital_dataset", exist_ok=True)
for idx, seed in enumerate([101,202,303,404,505], start=1):
    G = build_hospital_scenario(seed=seed)
    path = f"hospital_dataset/hospital_v{idx:02d}.graphml"
    nx.write_graphml(G, path)
    print(f"✔ Saved scenario {idx} ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges) → {path}")


✔ Saved scenario 1 (213 nodes, 359 edges) → hospital_dataset/hospital_v01.graphml
✔ Saved scenario 2 (213 nodes, 346 edges) → hospital_dataset/hospital_v02.graphml
✔ Saved scenario 3 (213 nodes, 357 edges) → hospital_dataset/hospital_v03.graphml
✔ Saved scenario 4 (213 nodes, 360 edges) → hospital_dataset/hospital_v04.graphml
✔ Saved scenario 5 (213 nodes, 347 edges) → hospital_dataset/hospital_v05.graphml
